In [ ]:
# ==========================================
# 1. SETUP GOOGLE COLAB & MOUNT DRIVE
# ==========================================
from google.colab import drive
import os
drive.mount('/content/drive')
directory = '/content/drive/My Drive/Kaggle/IQ-OTH-NCCD/IQ-OTHNCCD-dataset/'
categories = ['Bengin cases', 'Malignant cases', 'Normal cases']

In [ ]:
# ==========================================
# 2. SETUP & CLINICAL DATA PREPROCESSING
# ==========================================
import os
import numpy as np
import cv2
import random
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils import class_weight

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input

# Ensure GPU memory growth to prevent OOM errors
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

img_size = 224 # Native input size for EfficientNet-B0

# Data Loading to ensure 3 channels (RGB)
def load_clinical_data():
    data = []
    for i in categories:
        path = os.path.join(directory, i)
        class_num = categories.index(i)
        for file in os.listdir(path):
            try:
                filepath = os.path.join(path, file)
                img = cv2.imread(filepath) # Automatically loads as 3 channels
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (img_size, img_size))
                # IMPORTANT: Use the model's native preprocessing instead of /255.0
                img = preprocess_input(img)
                data.append([img, class_num])
            except: pass
    random.shuffle(data)
    X = np.array([item[0] for item in data])
    y = np.array([item[1] for item in data])
    return X, y

X, y = load_clinical_data()
# 80/20 split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Address Class Imbalance
weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(enumerate(weights))

In [ ]:
# ==========================================
# 3. ARCHITECTURAL MODIFICATION (EFFICIENTNET)
# ==========================================

# Update the Model to allow "Unfreezing" earlier
def build_optimized_efficientnet(input_shape=(224, 224, 3), num_classes=3):
    base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=input_shape)

    # Start with base_model trainable to allow the Squeeze-and-Excitation blocks to adapt
    base_model.trainable = True
    # Freeze only the first 50 layers, letting the deeper clinical features adapt
    for layer in base_model.layers[:50]:
        layer.trainable = False

    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.BatchNormalization(),
        layers.Dropout(0.4), # Increased dropout to prevent overfitting on 996 images
        layers.Dense(128, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

# Use a slightly higher initial Learning Rate
model = build_optimized_efficientnet()
model.compile(optimizer=optimizers.Adam(learning_rate=1e-3),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
# ==========================================
# 4. SCIENTIFIC TRAINING PROTOCOL
# ==========================================
# Justifying epoch count with Early Stopping
train_callbacks = [
    callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=4)
]

print("Training EfficientNet-B0 with Clinical Constraints...")
history = model.fit(X_train, y_train, epochs=50, validation_split=0.1,
                    class_weight=class_weights_dict, callbacks=train_callbacks, batch_size=16)

# **XAI VALIDATION (GRAD-CAM)**

In [ ]:
# ==========================================
# 5. XAI VALIDATION (GRAD-CAM)
# ==========================================
#
def get_gradcam_heatmap(img_array, model, last_conv_layer_name):
    # 1. Get the base EfficientNetB0 layer from the sequential model. It's the first layer.
    efficientnet_base_layer = model.get_layer('efficientnetb0')

    # 2. Create a model that takes the base EfficientNet's input and outputs the target conv layer's features
    # and the base EfficientNet's output. This is to extract features from the base model itself.
    feature_extractor = models.Model(
        inputs=efficientnet_base_layer.input,
        outputs=[efficientnet_base_layer.get_layer(last_conv_layer_name).output, efficientnet_base_layer.output]
    )

    # 3. Create a model that takes the output of the base EfficientNet and outputs the final predictions.
    # This is classifier head.
    classifier_input = layers.Input(shape=efficientnet_base_layer.output.shape[1:])
    x = classifier_input
    for layer in model.layers[1:]:
        x = layer(x)
    classifier_model = models.Model(inputs=classifier_input, outputs=x)

    with tf.GradientTape() as tape:
        inputs = tf.cast(img_array, tf.float32)

        # Get the feature map and the output of the base model
        last_conv_layer_output, base_output = feature_extractor(inputs)
        tape.watch(last_conv_layer_output)

        # Get the final predictions from the classifier head
        preds = classifier_model(base_output)
        class_channel = preds[:, tf.argmax(preds[0])]

    grads = tape.gradient(class_channel, last_conv_layer_output)

    # 4. Handle the None gradient error check
    if grads is None:
        raise ValueError("Gradient is still None. Check if 'base_model.trainable' is True.")

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = last_conv_layer_output[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

# ==========================================
# UPDATED VISUALIZATION BLOCK
# ==========================================
sample_idx = 5
sample_img = np.expand_dims(X_test[sample_idx], axis=0)

# IMPORTANT: In EfficientNetB0, 'top_activation' is the final conv layer.
# If that fails, try 'block7a_project_conv'
heatmap = get_gradcam_heatmap(sample_img, model, 'top_activation')

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.imshow(X_test[sample_idx]); plt.title('Original CT')
plt.subplot(1, 2, 2);
plt.imshow(X_test[sample_idx])
# Resize heatmap to match original image size
resized_heatmap = cv2.resize(heatmap, (img_size, img_size))
plt.imshow(resized_heatmap, alpha=0.5, cmap='jet')
plt.title('XAI Validation (Grad-CAM)')
plt.show()

# Test XAI on a Malignant Sample
sample_idx = 5
sample_img = np.expand_dims(X_test[sample_idx], axis=0)
# 'top_activation' is the typical final feature map for EfficientNet
heatmap = get_gradcam_heatmap(sample_img, model, 'top_activation')

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.imshow(X_test[sample_idx]); plt.title('Original CT')
plt.subplot(1, 2, 2); plt.imshow(X_test[sample_idx]);
plt.imshow(cv2.resize(heatmap, (img_size, img_size)), alpha=0.5, cmap='jet')
plt.title('Clinical XAI (Grad-CAM)')
plt.show()

# **FINAL COMPARATIVE EVALUATION**

In [ ]:
# ==========================================
# 6. FINAL COMPARATIVE EVALUATION
# ==========================================
y_pred = np.argmax(model.predict(X_test), axis=1)
print("\n--- Final Performance Metrics ---")
print(classification_report(y_test, y_pred, target_names=categories))

# **STATISTICAL PERFORMANCE ANALYSIS**

In [ ]:
# ==========================================
# 7. STATISTICAL PERFORMANCE ANALYSIS
# ==========================================
# Comparing latency (ms/image)
import time

start_time = time.time()
eff_preds = model.predict(X_test)
end_time = time.time()

avg_latency = (end_time - start_time) / len(X_test) * 1000
print(f"EfficientNet-B0 Average Latency: {avg_latency:.2f} ms/image")

# **EfficientNet-B0 Confusion Matrix**

In [ ]:
# ==========================================
# 8. CONFUSSION MATRIX
# ==========================================
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Generate predictions for the test set
print("Generating predictions for EfficientNet-B0...")
effnet_probs = model.predict(X_test, verbose=1)
effnet_preds = np.argmax(effnet_probs, axis=1)

# Create the confusion matrix
cm = confusion_matrix(y_test, effnet_preds)

# Plotting the Confusion Matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=categories, yticklabels=categories)

plt.title('Confusion Matrix: EfficientNet-B0 Lung Cancer Classification')
plt.ylabel('Actual Clinical Diagnosis')
plt.xlabel('AI Predicted Diagnosis')
plt.tight_layout()
plt.show()

# Print the full classification report
print("\n--- Final Performance Metrics (EfficientNet-B0) ---")
print(classification_report(y_test, effnet_preds, target_names=categories))

# **ROC Curve vs Precision-Recall Curve**

In [ ]:
# ==========================================
# 9. PERFORMANCE ANALYSIS - ROC Vs Precision
# ==========================================
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
from sklearn.preprocessing import label_binarize

# Generate predictions for the test set
print("Generating predictions for EfficientNet-B0...")
effnet_probs = model.predict(X_test, verbose=1)

# Binarize labels for multi-class evaluation
# Classes: 0: Benign, 1: Malignant, 2: Normal
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
categories = ['Benign cases', 'Malignant cases', 'Normal cases']

# Setup Plotting Environment
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Generate Curves for each class
colors = ['blue', 'red', 'green']

for i, color in enumerate(colors):
    # --- ROC Curve Calculation ---
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], effnet_probs[:, i])
    roc_auc = auc(fpr, tpr)
    ax1.plot(fpr, tpr, color=color, lw=2,
             label=f'{categories[i]} (AUC = {roc_auc:.4f})')

    # --- Precision-Recall Curve Calculation ---
    precision, recall, _ = precision_recall_curve(y_test_bin[:, i], effnet_probs[:, i])
    ap = average_precision_score(y_test_bin[:, i], effnet_probs[:, i])
    ax2.plot(recall, precision, color=color, lw=2,
             label=f'{categories[i]} (AP = {ap:.4f})')

# 5. Format ROC Plot
ax1.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax1.set_xlim([0.0, 1.0])
ax1.set_ylim([0.0, 1.05])
ax1.set_xlabel('False Positive Rate (1 - Specificity)')
ax1.set_ylabel('True Positive Rate (Sensitivity)')
ax1.set_title('ROC Curve: EfficientNet-B0 Performance')
ax1.legend(loc="lower right")
ax1.grid(alpha=0.3)

# 6. Format Precision-Recall Plot
ax2.set_xlim([0.0, 1.0])
ax2.set_ylim([0.0, 1.05])
ax2.set_xlabel('Recall (Sensitivity)')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curve: EfficientNet-B0 Performance')
ax2.legend(loc="lower left")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# **DECISION CURVE ANALYSIS (DCA) vs ROC CURVE**

In [ ]:
# ==========================================
# 10. PERFORMANCE ANALYSIS - DCA Vs ROC
# ==========================================
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

# Generate predictions for the test set
print("Generating predictions for EfficientNet-B0...")
effnet_probs = model.predict(X_test, verbose=1)

# Binarize labels for multi-class evaluation (Class 1 is Malignant)
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
y_true_mal = y_test_bin[:, 1]
y_prob_mal = effnet_probs[:, 1]

# Setup Plotting Environment
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# --- PLOT 1: ROC CURVE ---
fpr, tpr, _ = roc_curve(y_true_mal, y_prob_mal)
roc_auc = auc(fpr, tpr)
ax1.plot(fpr, tpr, color='green', lw=2, label=f'Malignant Detection (AUC = {roc_auc:.4f})')
ax1.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax1.set_xlabel('False Positive Rate (1 - Specificity)')
ax1.set_ylabel('True Positive Rate (Sensitivity)')
ax1.set_title('ROC Curve: EfficientNet-B0 Diagnostic Accuracy')
ax1.legend(loc="lower right")
ax1.grid(alpha=0.3)

# --- PLOT 2: DECISION CURVE ANALYSIS (DCA) ---
def calculate_net_benefit(y_true, y_prob, thresh):
    y_pred = (y_prob >= thresh)
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    n = len(y_true)
    if n == 0 or thresh == 1: return 0
    return (tp / n) - (fp / n) * (thresh / (1 - thresh))

thresholds = np.linspace(0.01, 0.99, 100)
net_benefit_model = [calculate_net_benefit(y_true_mal, y_prob_mal, t) for t in thresholds]

prevalence = np.mean(y_true_mal)
net_benefit_all = [prevalence - (1 - prevalence) * (t / (1 - t)) for t in thresholds]

ax2.plot(thresholds, net_benefit_model, color='green', lw=2, label='EfficientNet-B0')
ax2.plot(thresholds, net_benefit_all, 'k--', alpha=0.3, label='Treat All')
ax2.axhline(y=0, color='black', lw=1, label='Treat None')

ax2.set_ylim([-0.05, prevalence + 0.1])
ax2.set_xlabel('Threshold Probability')
ax2.set_ylabel('Net Benefit')
ax2.set_title('Decision Curve Analysis (DCA): Clinical Utility')
ax2.legend(loc="upper right")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# **Cost-Sensitive Learning and Dynamic Callbacks**

In [ ]:
# ==========================================
# 11. Class Weight
# ==========================================
from sklearn.utils import class_weight
from tensorflow.keras import optimizers, callbacks
import numpy as np

# Mathematically compute class weights to penalize minority misclassifications
weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
clinical_class_weights = dict(enumerate(weights))

# Implement Early Stopping to justify epoch counts and prevent overtraining
train_callbacks = [
    callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=4)
]

# Model compilation targeting sparse categorical crossentropy
model.compile(optimizer=optimizers.Adam(learning_rate=1e-3),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

print("--- Clinical Class Weights ---")
for i, weight in enumerate(weights):
    print(f"Class {i} ({categories[i]}): {weight:.4f}")

print("\n--- Model Compilation Summary ---")
print(f"Optimizer: {model.optimizer.__class__.__name__}")
print(f"Initial Learning Rate: {model.optimizer.learning_rate.numpy()}")
print(f"Loss Function: {model.loss}")

# **Training and validation history: accuracy vs loss**

In [ ]:
# ==========================================
# 12. Training and Validation
# ==========================================
import matplotlib.pyplot as plt

# Define the function to plot the history
def plot_effnet_performance(history):
    """
    Takes the history object from model.fit() and plots
    Accuracy and Loss for training and validation.
    """
    # Create a figure with two side-by-side plots
    plt.figure(figsize=(14, 5))

    # --- PLOT 1: ACCURACY ---
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Training Accuracy', color='green', linestyle='--')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy', color='darkgreen')
    plt.title('EfficientNet-B0: Training & Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy Score')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # --- PLOT 2: LOSS ---
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Training Loss', color='red', linestyle='--')
    plt.plot(history.history['val_loss'], label='Validation Loss', color='darkred')
    plt.title('EfficientNet-B0: Training & Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Cross-Entropy Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# RUN THE PLOT
plot_effnet_performance(history)